In [ ]:
import polars as pl
from pathlib import Path


def direction(lo, hi):
    """Classify a 95% CI on a fold/ratio scale relative to the null (1.0)."""
    if lo is None or hi is None:
        return None
    if lo > 1.0:
        return "increased"
    if hi < 1.0:
        return "decreased"
    return "n.s."

In [ ]:
# Assuming either all *_compressed.tar.zst files or all_groups_lean.tar.zst file is/are decompressed
CWD = Path().resolve()
BASE = CWD.parent

print(f"Current working directory (where figure will be saved): {CWD}")
print(f"Folder where groups (data) folder should be: {BASE}")

In [ ]:
groups = [
    "fungi_mit",
    "metazoans_mit",
    "plants_mit",
    "plants_plt",
    "protists_mit",
    "protists_plt",
    "green_algae_mit",
    "green_algae_plt",
]
label_map = {
    "fungi_mit": "Fungi (mitochondria)",
    "green_algae_mit": "Green algae (mitochondria)",
    "metazoans_mit": "Metazoans (mitochondria)",
    "plants_mit": "Plants (mitochondria)",
    "protists_mit": "Protists (mitochondria)",
    "green_algae_plt": "Green algae (plastid)",
    "plants_plt": "Plants (plastid)",
    "protists_plt": "Protists (plastid)",
}

In [ ]:
rows_s4 = []

for g in groups:
    gdir = BASE / g

    h2_row = pl.read_csv(
        gdir / "brms_type_length" / "brms_sigma_h2_results_row.tsv", separator="\t"
    ).row(0, named=True)

    rows_s4.append(
        {
            "group": label_map[g],
            "igr_regions": h2_row["N_regions"],
            "N_taxa": h2_row["N_taxa"],
            "sigma_ratio_conv": h2_row["sigma_ratio_conv"],
            "sigma_ratio_conv_lo": h2_row["sigma_ratio_conv_lo"],
            "sigma_ratio_conv_hi": h2_row["sigma_ratio_conv_hi"],
            "conv_dispersion": direction(
                h2_row["sigma_ratio_conv_lo"], h2_row["sigma_ratio_conv_hi"]
            ),
            "sigma_ratio_div": h2_row["sigma_ratio_div"],
            "sigma_ratio_div_lo": h2_row["sigma_ratio_div_lo"],
            "sigma_ratio_div_hi": h2_row["sigma_ratio_div_hi"],
            "div_dispersion": direction(
                h2_row["sigma_ratio_div_lo"], h2_row["sigma_ratio_div_hi"]
            ),
            "sigma_baseline_sd": h2_row["sigma_baseline_sd"],
            "H2_len": h2_row["H2_len_median"],
            "H2_len_lo": h2_row["H2_len_lo"],
            "H2_len_hi": h2_row["H2_len_hi"],
            "H2_type": h2_row["H2_type_median"],
            "H2_type_lo": h2_row["H2_type_lo"],
            "H2_type_hi": h2_row["H2_type_hi"],
            "H2_both": h2_row["H2_both_median"],
            "H2_both_lo": h2_row["H2_both_lo"],
            "H2_both_hi": h2_row["H2_both_hi"],
        }
    )

s4 = pl.DataFrame(rows_s4)

float_cols = [col for col in s4.columns if s4[col].dtype in [pl.Float32, pl.Float64]]
s4 = s4.with_columns([pl.col(col).round(3) for col in float_cols])

s4.write_csv(BASE / "code" / "supplemental_table4.tsv", separator="\t")